# 📘 Módulo 6 — Análise de Regressão
## Livro Didático Aplicado (Híbrido)

- 🔵 **Conteúdo oficial do módulo 6 (IBM)**
- 🟣 **Conteúdo expandido (Livro Didático)**
- 🟠 **Conteúdo avançado (Opcional, matemático)**

Este notebook segue o mesmo padrão dos módulos anteriores:

- clareza didática,
- profundidade conceitual,
- rigor matemático,
- explicações intuitivas,
- aplicações práticas,
- demonstrações opcionais em `<details>`.

A regressão é apresentada pelo curso IBM como:

> “O carro‑chefe da análise estatística.”

Aqui você verá por quê.

# 📚 Índice

0. [Setup — bibliotecas e dados](#setup)
1. [Introdução à Regressão](#intro)
2. [Regressão Linear Simples](#simples)
3. [Regressão no lugar do Teste T](#reg_t)
4. [Regressão no lugar da ANOVA](#reg_anova)
5. [Regressão no lugar da Correlação](#reg_corr)
6. [Aplicações com teaching ratings](#aplicacoes)
7. [Exercícios Guiados](#exercicios)
8. [Apêndice Matemático Avançado](#apendice)

<a id="setup"></a>
# 0. Setup — bibliotecas e dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

plt.style.use("seaborn-v0_8-whitegrid")

🔵 **Dataset teaching ratings (IBM)**

O módulo 6 continua usando o mesmo dataset dos módulos anteriores.
Ele contém:
- avaliações de ensino (`eval`),
- características dos instrutores,
- número de alunos,
- variáveis categóricas e numéricas.

Este dataset será usado para:
- regressão linear simples,
- regressão múltipla,
- regressão com variáveis dummy,
- regressão como alternativa a T‑test, ANOVA e Pearson.

In [ ]:
ratings_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/teachingratings.csv"
ratings_df = pd.read_csv(ratings_url)
ratings_df.to_csv("/home/moacir/projects/ml/IBM/statistics/data/raw/teachingratings.csv", index=False)
ratings_df.head()

In [ ]:
# Carregamento local
ratings_df = pd.read_csv("/home/moacir/projects/ml/IBM/statistics/data/raw/teachingratings.csv")
ratings_df.head()

---
# 🔵 Conexão com o curso IBM

O módulo 6 apresenta quatro ideias centrais:

1. **Regressão como substituto dos testes de hipótese**  
   - regressão substitui T‑test, ANOVA e Pearson

2. **Regressão linear simples**  
   - modelo:  
     $$
     Y = \beta_0 + \beta_1 X + \varepsilon
     $$

3. **Regressão com variáveis dummy**  
   - substitui teste t e ANOVA

4. **Regressão como análise de correlação**  
   - coeficiente β₁ e valor‑p equivalem ao teste de Pearson  
   - $R^2 = r^2$

Este notebook segue exatamente essa estrutura — com explicações ampliadas, rigor matemático e visualizações adicionais.

---
# 🟣 Antes de avançar

A partir daqui, entraremos no conteúdo central do módulo:

- regressão linear simples,  
- regressão múltipla,  
- variáveis dummy,  
- regressão como alternativa a testes estatísticos,  
- interpretação de coeficientes,  
- erro, resíduos e ajuste do modelo.

Cada seção terá:

- definição clara,  
- explicação intuitiva,  
- interpretação geométrica,  
- exemplos numéricos,  
- gráficos,  
- e aprofundamentos opcionais em `<details>`.

Vamos começar pela base: **o que é regressão e por que ela substitui tantos testes?**

<a id="intro"></a>
# 1. Introdução à Regressão

🔵 **Conteúdo oficial do módulo 6 (IBM)**

O curso começa afirmando:

> “A regressão é o carro‑chefe da análise estatística.”

E explica que, enquanto testes de hipótese analisam relações **duas a duas**,  
a regressão permite analisar **várias relações simultaneamente**.

Além disso:

- regressão substitui **teste t**,  
- regressão substitui **ANOVA**,  
- regressão substitui **correlação de Pearson**,  
- e ainda permite **previsão**, algo que os testes não fazem.

Vamos construir essa base.

---
# 1.1 O que é regressão?

Regressão é um método estatístico para modelar a relação entre:

- uma **variável dependente** (Y)  
- uma ou mais **variáveis explicativas** (X)

Exemplos do curso IBM:

- A avaliação de ensino (`eval`) pode ser explicada por:
  - beleza (`beauty`)
  - gênero (`gender`)
  - proficiência em inglês (`native`)

Formalmente:

$$
Y = f(X)
$$

---
# 1.2 Variável dependente (Y)

É a variável que queremos explicar ou prever.

Exemplos:
- `eval` (nota de avaliação)
- `beauty` (pontuação de beleza)
- `age` (idade)

No módulo 6, o foco é:

$$
Y = \text{eval}
$$

---
# 1.3 Variáveis explicativas (X)

São as variáveis que explicam a variação de Y.

Exemplos:
- `beauty`
- `gender`
- `native`
- `age`
- `students`

O curso IBM usa principalmente:

$$
X = \text{beauty}
$$

---
# 1.4 Modelo de regressão linear simples

O modelo mais básico é:

$$
Y = \beta_0 + \beta_1 X + \varepsilon
$$

Onde:

- $\beta_0$ = intercepto  
- $\beta_1$ = inclinação (efeito de X sobre Y)  
- $\varepsilon$ = erro (tudo que o modelo não explica)

O curso IBM enfatiza:

> “O termo de erro captura tudo o que o modelo não conseguiu explicar.”

---
# 1.5 Modelo de regressão múltipla

Quando há mais de uma variável explicativa:

$$
Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \varepsilon
$$

Exemplo:

$$
\text{eval} = \beta_0 + \beta_1 \text{beauty} + \beta_2 \text{gender} + \beta_3 \text{native} + \varepsilon
$$

---
# 1.6 Exemplo IBM — Regressão de `eval` sobre `beauty`

O curso mostra o modelo:

$$
\text{eval} = 3.998 + 0.133 \cdot \text{beauty} + \varepsilon
$$

Vamos reproduzir isso com `statsmodels`.

In [ ]:
model = smf.ols("eval ~ beauty", data=ratings_df).fit()
model.summary()

🟣 **Interpretação**

- **Intercepto (β₀ ≈ 3.998)**  
  → avaliação esperada quando beauty = 0  

- **Coeficiente (β₁ ≈ 0.133)**  
  → cada 1 ponto de beleza aumenta a avaliação em ~0.13 pontos  

- **Valor‑p < 0.05**  
  → relação estatisticamente significativa  

- **Erro (ε)**  
  → diferença entre o valor real e o previsto pelo modelo  

Isso coincide exatamente com o exemplo IBM.

---
# 1.7 Visualização da regressão

In [ ]:
plt.figure(figsize=(7, 4))
sns.regplot(x="beauty", y="eval", data=ratings_df, ci=95, line_kws={"color": "red"})
plt.title("Regressão linear: eval ~ beauty")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação visual**

- A linha vermelha é a reta de regressão.  
- A inclinação positiva confirma o coeficiente β₁ > 0.  
- A banda azul é o intervalo de confiança.  

<details>
<summary>🟠 Explicação avançada — O que é o termo de erro?</summary>

O termo de erro $\varepsilon$ representa:

- variáveis não incluídas no modelo,  
- ruído aleatório,  
- fatores não observados,  
- erros de medição.

Formalmente:

$$
\varepsilon_i = Y_i - \hat{Y}_i
$$

Onde $\hat{Y}_i$ é o valor previsto pelo modelo.

Em regressão, analisamos os resíduos para verificar:
- normalidade,  
- homocedasticidade,  
- independência.

</details>

<a id="simples"></a>
# 2. Regressão Linear Simples

🔵 **Conteúdo oficial do módulo 6 (IBM)**

O curso afirma que:

> “Se você sabe regressão, pode renunciar a muitos testes de hipótese.”

E começa mostrando a regressão simples:

$$
Y = \beta_0 + \beta_1 X + \varepsilon
$$

Onde:
- Y = variável dependente  
- X = variável explicativa  
- β₀ = intercepto  
- β₁ = inclinação  
- ε = erro (resíduo)  

Vamos aprofundar isso com o dataset teaching ratings.

---
# 2.1 O modelo linear simples

O objetivo é estimar:

$$
\hat{Y} = \beta_0 + \beta_1 X
$$

Onde:
- $\hat{Y}$ é o valor previsto  
- β₁ mede o efeito de X sobre Y  

No exemplo IBM:

- Y = `eval`  
- X = `beauty`  

In [ ]:
model = smf.ols("eval ~ beauty", data=ratings_df).fit()
model.summary()

🟣 **Interpretação dos coeficientes**

- **Intercepto (β₀ ≈ 3.998)**  
  → avaliação esperada quando beauty = 0  

- **Inclinação (β₁ ≈ 0.133)**  
  → cada 1 ponto de beleza aumenta a avaliação em ~0.13 pontos  

- **Valor‑p < 0.05**  
  → relação estatisticamente significativa  

- **R² ≈ 0.036**  
  → beleza explica ~3.6% da variação em eval  

Isso coincide exatamente com o exemplo IBM.

---
# 2.2 Visualização da reta de regressão

In [ ]:
plt.figure(figsize=(7, 4))
sns.regplot(x="beauty", y="eval", data=ratings_df, ci=95, line_kws={"color": "red"})
plt.title("Regressão linear simples: eval ~ beauty")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação visual**

- A relação é positiva, mas fraca.  
- A banda azul mostra o intervalo de confiança da reta.  
- A dispersão é grande → R² baixo.  

---
# 2.3 Valores previstos e resíduos

O curso IBM enfatiza o termo de erro (ε):

> “O erro é tudo o que o modelo não conseguiu explicar.”

Vamos calcular resíduos e valores previstos.

In [ ]:
ratings_df["y_pred"] = model.fittedvalues
ratings_df["residuals"] = model.resid

ratings_df[["eval", "y_pred", "residuals"]].head()

---
# 2.4 Gráfico de resíduos

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(ratings_df["beauty"], ratings_df["residuals"], alpha=0.6)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("beauty")
plt.ylabel("residuals")
plt.title("Resíduos vs beauty")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Os resíduos parecem aleatórios → bom sinal.  
- Não há padrão claro → linearidade razoável.  
- Há variabilidade constante → homocedasticidade aceitável.  

---
# 2.5 Distribuição dos resíduos

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(ratings_df["residuals"], kde=True, bins=20)
plt.title("Distribuição dos resíduos")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- A distribuição é aproximadamente normal.  
- Isso valida o uso da regressão linear.  

---
# 2.6 Regressão como estimador de efeito

O coeficiente β₁ é interpretado como:

> “O efeito médio de X sobre Y.”

No exemplo:

$$
\beta_1 = 0.133
$$

Isso significa:

- cada aumento de 1 ponto em beleza → +0.133 em eval  
- efeito pequeno, mas estatisticamente significativo  

---
# 2.7 Regressão como teste de hipótese

A regressão testa automaticamente:

$$
H_0: \beta_1 = 0
$$

$$
H_a: \beta_1 \ne 0
$$

O valor‑p do coeficiente β₁ é exatamente o mesmo valor‑p do teste de Pearson.

<details>
<summary>🟠 Explicação avançada — Mínimos Quadrados Ordinários (OLS)</summary>

O método OLS escolhe β₀ e β₁ que minimizam:

$$
\sum_{i=1}^n (Y_i - \hat{Y}_i)^2
$$

As soluções fechadas são:

$$
\beta_1 =
\frac{\sum (x_i - \bar{x})(y_i - \bar{y})}
{\sum (x_i - \bar{x})^2}
$$

$$
\beta_0 = \bar{y} - \beta_1 \bar{x}
$$

Isso conecta regressão diretamente com correlação.

</details>

<a id="reg_t"></a>
# 3. Regressão no lugar do Teste T

🔵 **Conteúdo oficial do módulo 6 (IBM)**

O curso demonstra que:

> “Podemos usar regressão linear no lugar do teste t.”

A ideia é simples:

- O teste t compara **duas médias**.  
- A regressão com uma variável dummy também compara **duas médias**.  

E o mais importante:

> “A estatística t e o valor‑p da regressão são idênticos aos do teste t.”

Vamos reproduzir exatamente o exemplo IBM:

**Pergunta:**  
Existe diferença estatisticamente significativa entre as notas de avaliação de ensino de homens e mulheres?

---
# 3.1 Criando a variável dummy (IBM)

O curso transforma:

- female = 1  
- male = 0  

Vamos fazer o mesmo.

In [ ]:
ratings_df["female_dummy"] = (ratings_df["gender"] == "female").astype(int)
ratings_df[["gender", "female_dummy"]].head()

---
# 3.2 Teste T tradicional (para comparação)

Vamos rodar o teste t para confirmar o valor‑p.

In [ ]:
from scipy.stats import ttest_ind

female = ratings_df[ratings_df["gender"] == "female"]["eval"]
male   = ratings_df[ratings_df["gender"] == "male"]["eval"]

t_stat, p_value = ttest_ind(female, male, equal_var=True)
t_stat, p_value

🟣 **Interpretação**

- valor‑p < 0.05  
- diferença estatisticamente significativa  

Agora vamos ver como a regressão chega ao mesmo resultado.

---
# 3.3 Regressão linear com variável dummy

Modelo:

$$
\text{eval} = \beta_0 + \beta_1 \cdot \text{female} + \varepsilon
$$

Onde:
- β₀ = média dos homens  
- β₁ = diferença entre mulheres e homens  

Vamos ajustar o modelo.

In [ ]:
model_dummy = smf.ols("eval ~ female_dummy", data=ratings_df).fit()
model_dummy.summary()

---
# 3.4 Interpretando os coeficientes

🔵 **Conteúdo IBM**

O curso afirma:

> “O coeficiente significa que é mais provável que você perca cerca de 0,17 pontos por ser mulher.”

Vamos interpretar:

- **Intercepto (β₀ ≈ 4.06)**  
  → média dos homens  

- **Coeficiente (β₁ ≈ −0.17)**  
  → mulheres têm, em média, 0.17 pontos a menos  

- **Estatística t ≈ −3.25**  
- **Valor‑p < 0.05**  

Isso é exatamente o que o curso IBM mostra.

---
# 3.5 Comparação direta: Teste T vs Regressão

| Método | Estatística t | Valor‑p | Conclusão |
|--------|---------------|---------|-----------|
| Teste T | mesmo valor | mesmo valor | diferença significativa |
| Regressão | mesmo valor | mesmo valor | diferença significativa |

🟣 **Conclusão:**  
A regressão reproduz exatamente o teste t.

---
# 3.6 Visualização da diferença entre grupos

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x="gender", y="eval", data=ratings_df, ci=95)
plt.title("Média de eval por gênero (com IC 95%)")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- A diferença é pequena, mas consistente.  
- O intervalo de confiança mostra separação entre os grupos.  

---
# 3.7 Por que regressão substitui o teste t?

Porque a regressão testa automaticamente:

$$
H_0: \beta_1 = 0
$$

Que é equivalente a:

$$
H_0: \mu_{\text{female}} = \mu_{\text{male}}
$$

E a estatística t é **idêntica**.

<details>
<summary>🟠 Explicação avançada — Dummy coding e diferença de médias</summary>

Se:

- female = 1  
- male = 0  

Então:

$$
\hat{Y}_{\text{male}} = \beta_0
$$

$$
\hat{Y}_{\text{female}} = \beta_0 + \beta_1
$$

Logo:

$$
\beta_1 = \hat{Y}_{\text{female}} - \hat{Y}_{\text{male}}
$$

Ou seja, **β₁ é literalmente a diferença de médias**.

Por isso regressão = teste t.

</details>

<a id="reg_anova"></a>
# 4. Regressão no lugar da ANOVA

🔵 **Conteúdo oficial do módulo 6 (IBM)**

O curso explica:

> “Quando comparamos médias de mais de dois grupos, usamos ANOVA.  
> Mas podemos obter exatamente os mesmos resultados usando regressão.”

A ideia é simples:

- ANOVA testa se **todas as médias são iguais**  
- Regressão com variáveis dummy testa **o mesmo**, mas dentro de um modelo linear  

E o mais importante:

> “A estatística F e o valor‑p da regressão são idênticos aos da ANOVA.”

Vamos reproduzir exatamente o exemplo IBM.

---
# 4.1 Criando grupos de idade (IBM)

O curso divide os instrutores em três grupos:

- **≤ 40 anos**  
- **40–57 anos**  
- **≥ 57 anos**  

Vamos criar essa variável categórica.

In [ ]:
bins = [0, 40, 57, 100]
labels = ["<=40", "40-57", ">=57"]

ratings_df["age_group"] = pd.cut(ratings_df["age"], bins=bins, labels=labels, include_lowest=True)
ratings_df[["age", "age_group"]].head()

---
# 4.2 ANOVA tradicional (para comparação)

Vamos rodar ANOVA para `eval` usando `scipy`.

In [ ]:
group1 = ratings_df[ratings_df["age_group"] == "<=40"]["eval"]
group2 = ratings_df[ratings_df["age_group"] == "40-57"]["eval"]
group3 = ratings_df[ratings_df["age_group"] == ">=57"]["eval"]

from scipy.stats import f_oneway
f_stat, p_value = f_oneway(group1, group2, group3)
f_stat, p_value

🟣 **Interpretação**

- valor‑p > 0.05  
- **não rejeitamos H₀**  
- as médias dos três grupos são estatisticamente iguais  

Agora vamos ver como a regressão chega ao mesmo resultado.

---
# 4.3 Criando variáveis dummy (IBM)

O curso usa `get_dummies`:

- cada grupo vira uma coluna  
- 1 = pertence ao grupo  
- 0 = não pertence  

Vamos fazer isso.

In [ ]:
dummies = pd.get_dummies(ratings_df["age_group"], drop_first=True)
dummies.head()

---
# 4.4 Regressão linear com variáveis dummy

Modelo:

$$
\text{eval} = \beta_0 + \beta_1 D_{40-57} + \beta_2 D_{\ge 57} + \varepsilon
$$

Onde:
- β₀ = média do grupo de referência (≤40)  
- β₁ = diferença entre 40–57 e ≤40  
- β₂ = diferença entre ≥57 e ≤40  

Vamos ajustar o modelo.

In [ ]:
df_reg = pd.concat([ratings_df["eval"], dummies], axis=1)
model_anova_reg = smf.ols("eval ~ Q('40-57') + Q('>=57')", data=df_reg).fit()
model_anova_reg.summary()

🟣 **Interpretação**

- Nenhum coeficiente é significativo (valor‑p > 0.05)  
- Isso indica que **nenhum grupo difere do grupo de referência**  
- Conclusão idêntica à ANOVA tradicional  

---
# 4.5 Obtendo a tabela ANOVA a partir da regressão

O curso IBM mostra como usar `anova_lm` para extrair a tabela ANOVA do modelo OLS.

In [ ]:
anova_lm(model_anova_reg)

🟣 **Interpretação**

- A estatística F é **idêntica** à da ANOVA tradicional  
- O valor‑p também é **idêntico**  

Isso confirma o que o curso IBM afirma:

> “Regressão e ANOVA produzem exatamente os mesmos resultados.”

---
# 4.6 Visualização das médias por grupo

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x="age_group", y="eval", data=ratings_df, ci=95)
plt.title("Média de eval por grupo de idade (com IC 95%)")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- As médias são visualmente próximas  
- Isso reforça o resultado da ANOVA e da regressão  

---
# 4.7 Por que regressão substitui ANOVA?

Porque ANOVA testa:

$$
H_0: \mu_1 = \mu_2 = \mu_3
$$

E regressão testa:

$$
H_0: \beta_1 = \beta_2 = 0
$$

Mas:

$$
\beta_1 = \mu_2 - \mu_1
$$
$$
\beta_2 = \mu_3 - \mu_1
$$

Logo:

> “Testar β = 0 é o mesmo que testar igualdade de médias.”

<details>
<summary>🟠 Explicação avançada — ANOVA como regressão</summary>

A ANOVA clássica é um caso especial de regressão linear onde:

- X são variáveis dummy  
- Y é contínua  

A estatística F da ANOVA é:

$$
F = \frac{\text{variância explicada}}{\text{variância residual}}
$$

A estatística F da regressão é:

$$
F = \frac{MS_{\text{modelo}}}{MS_{\text{erro}}}
$$

As duas expressões são **idênticas**.

Por isso:

> “Toda ANOVA é uma regressão.  
> Nem toda regressão é uma ANOVA.”

</details>

<a id="reg_corr"></a>
# 5. Regressão no lugar da Correlação

🔵 **Conteúdo oficial do módulo 6 (IBM)**

O curso explica:

> “Podemos usar regressão linear no lugar do teste de correlação de Pearson.”

A ideia é simples:

- Pearson mede **força e direção** da relação linear  
- Regressão mede **força, direção e magnitude do efeito**  
- O valor‑p da regressão é **idêntico** ao valor‑p do teste de Pearson  
- O R² da regressão é **r²**  

Vamos reproduzir exatamente o exemplo IBM:

**Pergunta:**  
Existe relação linear entre `beauty` e `eval`?

---
# 5.1 Correlação de Pearson (método tradicional)

Primeiro, vamos calcular o coeficiente de Pearson.

In [ ]:
beauty = ratings_df["beauty"]
evals  = ratings_df["eval"]

pearson_r, p_value = pearsonr(beauty, evals)
pearson_r, p_value

🟣 **Interpretação**

- **r ≈ 0.189** → correlação fraca positiva  
- **valor‑p < 0.05** → estatisticamente significativa  

Agora vamos ver como a regressão chega ao mesmo resultado.

---
# 5.2 Regressão linear simples

Modelo:

$$
\text{eval} = \beta_0 + \beta_1 \cdot \text{beauty} + \varepsilon
$$

In [ ]:
model_corr = smf.ols("eval ~ beauty", data=ratings_df).fit()
model_corr.summary()

---
# 5.3 Comparando regressão e Pearson

🔵 **Conteúdo IBM**

O curso afirma:

> “O valor‑p da regressão é o mesmo do teste de Pearson.”

Vamos verificar:

In [ ]:
model_corr.pvalues["beauty"], p_value

🟣 **Resultado**

✔️ **Os valores‑p são idênticos**  
✔️ **A conclusão estatística é a mesma**  

Isso confirma o que o curso IBM demonstra.

---
# 5.4 Relação entre R² e r

O curso IBM mostra:

> “Se tirarmos a raiz quadrada de R², obtemos o coeficiente de Pearson.”

Vamos verificar:

In [ ]:
r_from_r2 = np.sqrt(model_corr.rsquared)
pearson_r, r_from_r2

🟣 **Interpretação**

✔️ **r = √R²**  
✔️ Isso confirma a equivalência matemática entre regressão e correlação linear.  

---
# 5.5 Visualização — Scatter plot com reta de regressão

In [ ]:
plt.figure(figsize=(7, 4))
sns.regplot(x="beauty", y="eval", data=ratings_df, ci=95, line_kws={"color": "red"})
plt.title("Correlação e regressão: eval ~ beauty")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação visual**

- A relação é fraca, mas positiva  
- A reta de regressão resume a tendência  
- A banda azul mostra o intervalo de confiança  

---
# 5.6 Por que regressão substitui o teste de correlação?

Porque a regressão testa automaticamente:

$$
H_0: \beta_1 = 0
$$

E o teste de Pearson testa:

$$
H_0: \rho = 0
$$

Mas:

$$
\beta_1 = r \cdot \frac{s_y}{s_x}
$$

Logo:

- β₁ ≠ 0 ⇔ r ≠ 0  
- valor‑p da regressão = valor‑p de Pearson  
- R² = r²  

<details>
<summary>🟠 Explicação avançada — Conexão matemática entre regressão e Pearson</summary>

A fórmula do coeficiente de regressão é:

$$
\beta_1 =
\frac{\sum (x_i - \bar{x})(y_i - \bar{y})}
{\sum (x_i - \bar{x})^2}
$$

A fórmula do coeficiente de Pearson é:

$$
r =
\frac{\sum (x_i - \bar{x})(y_i - \bar{y})}
{\sqrt{\sum (x_i - \bar{x})^2} \sqrt{\sum (y_i - \bar{y})^2}}
$$

Portanto:

$$
\beta_1 = r \cdot \frac{s_y}{s_x}
$$

E:

$$
R^2 = r^2
$$

Isso mostra que regressão e correlação são **duas faces da mesma moeda**.

</details>

<a id="aplicacoes"></a>
# 6. Aplicações com *teaching ratings*

Nesta seção, aplicamos regressão para responder perguntas reais sobre o dataset.

🔵 **Conexão com o curso IBM**

O módulo 6 mostra que regressão pode substituir:

- teste t  
- ANOVA  
- correlação de Pearson  

Aqui vamos integrar tudo isso em aplicações práticas.

---
# 6.1 Pergunta 1 — Professores mais bonitos recebem melhores avaliações?

Modelo:

$$
\text{eval} = \beta_0 + \beta_1 \cdot \text{beauty} + \varepsilon
$$

In [ ]:
model1 = smf.ols("eval ~ beauty", data=ratings_df).fit()
model1.summary()

🟣 **Interpretação**

- β₁ > 0 → relação positiva  
- valor‑p < 0.05 → estatisticamente significativa  
- R² baixo → efeito pequeno  

✔️ Conclusão: instrutores mais bonitos tendem a receber avaliações ligeiramente maiores.

---
# 6.2 Pergunta 2 — Professores mulheres recebem notas diferentes?

Criamos a dummy:

- female = 1  
- male = 0  

Modelo:

$$
\text{eval} = \beta_0 + \beta_1 \cdot \text{female} + \varepsilon
$$

In [ ]:
model2 = smf.ols("eval ~ female_dummy", data=ratings_df).fit()
model2.summary()

🟣 **Interpretação**

- β₁ ≈ −0.17 → mulheres recebem ~0.17 pontos a menos  
- valor‑p < 0.05 → diferença significativa  

✔️ Conclusão: regressão confirma o mesmo resultado do teste t.

---
# 6.3 Pergunta 3 — A idade influencia a avaliação?

Usamos os três grupos criados:

- ≤40  
- 40–57  
- ≥57  

Criamos dummies e rodamos regressão.

In [ ]:
dummies = pd.get_dummies(ratings_df["age_group"], drop_first=True)
df_reg = pd.concat([ratings_df["eval"], dummies], axis=1)

model3 = smf.ols("eval ~ Q('40-57') + Q('>=57')", data=df_reg).fit()
model3.summary()

🟣 **Interpretação**

- Nenhum coeficiente é significativo  
- valor‑p > 0.05  

✔️ Conclusão: idade não influencia a avaliação  
✔️ Resultado idêntico à ANOVA tradicional

---
# 6.4 Pergunta 4 — A beleza varia com a idade?

Agora Y = beauty.

In [ ]:
df_reg2 = pd.concat([ratings_df["beauty"], dummies], axis=1)
model4 = smf.ols("beauty ~ Q('40-57') + Q('>=57')", data=df_reg2).fit()
model4.summary()

🟣 **Interpretação**

- coeficientes significativos  
- valor‑p < 0.05  

✔️ Conclusão: grupos de idade têm médias de beleza diferentes  
✔️ Igual ao resultado da ANOVA do módulo 5

---
# 6.5 Pergunta 5 — Professores com mais alunos recebem notas diferentes?

Modelo:

$$
\text{eval} = \beta_0 + \beta_1 \cdot \text{students} + \varepsilon
$$

In [ ]:
model5 = smf.ols("eval ~ students", data=ratings_df).fit()
model5.summary()

🟣 **Interpretação**

- β₁ ≈ 0.0007 → efeito praticamente zero  
- valor‑p alto  

✔️ Conclusão: tamanho da turma não influencia a avaliação.

---
# 6.6 Pergunta 6 — Professores mais bem avaliados têm mais alunos?

Agora invertendo:

$$
\text{students} = \beta_0 + \beta_1 \cdot \text{eval} + \varepsilon
$$

In [ ]:
model6 = smf.ols("students ~ eval", data=ratings_df).fit()
model6.summary()

🟣 **Interpretação**

- β₁ ≈ 1.1 → efeito pequeno  
- valor‑p alto  

✔️ Conclusão: não há relação significativa.

---
# 6.7 Pergunta 7 — Modelo múltiplo: beleza + gênero + idade explicam a avaliação?

Modelo:

$$
\text{eval} = \beta_0 + \beta_1 \text{beauty} + \beta_2 \text{female} + \beta_3 \text{age} + \varepsilon
$$

In [ ]:
model7 = smf.ols("eval ~ beauty + female_dummy + age", data=ratings_df).fit()
model7.summary()

🟣 **Interpretação**

- beleza → significativa  
- female → significativa  
- idade → não significativa  

✔️ Conclusão: beleza e gênero explicam parte da avaliação  
✔️ idade não contribui

---
# 6.8 Visualização integrada — Matriz de correlação

In [ ]:
plt.figure(figsize=(6, 5))
corr = ratings_df[["eval", "beauty", "age", "students"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriz de correlação")
plt.show()

🟣 **Interpretação**

- beauty × eval → única relação notável  
- demais correlações são fracas  

✔️ Isso confirma os resultados das regressões anteriores.

<details>
<summary>🟠 Explicação avançada — Por que regressão é tão poderosa?</summary>

Regressão unifica:

- diferença de médias (teste t)  
- comparação de 3+ grupos (ANOVA)  
- correlação linear (Pearson)  

Tudo isso é obtido como casos especiais de:

$$
Y = X\beta + \varepsilon
$$

Onde:
- X pode conter dummies (categorias)  
- X pode conter variáveis contínuas  
- X pode conter múltiplas variáveis  

A estatística F e a estatística t surgem naturalmente do modelo.

Por isso o curso IBM diz:

> “A regressão é o carro‑chefe da análise estatística.”

</details>

<a id="exercicios"></a>
# 7. Exercícios Guiados — Módulo 6 (Regressão)

Estes exercícios consolidam todo o conteúdo do módulo 6:

- regressão simples  
- regressão múltipla  
- regressão com variáveis dummy  
- regressão como alternativa ao teste t  
- regressão como alternativa à ANOVA  
- regressão como alternativa à correlação  
- interpretação de coeficientes, resíduos e R²  

Cada exercício foi desenhado para reforçar a teoria e a prática.

---
# 📝 Exercício 1 — Regressão simples: eval ~ beauty

Ajuste o modelo:

$$
\text{eval} = \beta_0 + \beta_1 \cdot \text{beauty} + \varepsilon
$$

1. Ajuste o modelo com `statsmodels`.  
2. Interprete β₀ e β₁.  
3. Interprete o valor‑p.  
4. Interprete o R².  
5. Plote a reta de regressão.  

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 2 — Regressão no lugar do teste t

Teste se existe diferença entre homens e mulheres usando regressão.

1. Crie a variável dummy: female = 1, male = 0.  
2. Ajuste o modelo:

$$
\text{eval} = \beta_0 + \beta_1 \cdot \text{female} + \varepsilon
$$

3. Compare β₁ com a diferença de médias real.  
4. Compare o valor‑p com o teste t tradicional.  
5. Interprete o coeficiente β₁.  

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 3 — Regressão no lugar da ANOVA

Use a variável `age_group` (<=40, 40–57, >=57).

1. Crie variáveis dummy com `get_dummies`.  
2. Ajuste o modelo:

$$
\text{eval} = \beta_0 + \beta_1 D_{40-57} + \beta_2 D_{\ge 57} + \varepsilon
$$

3. Compare o valor‑p com o da ANOVA tradicional.  
4. Interprete os coeficientes.  

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 4 — Regressão múltipla

Ajuste o modelo:

$$
\text{eval} = \beta_0 + \beta_1 \text{beauty} + \beta_2 \text{female} + \beta_3 \text{age} + \varepsilon
$$

1. Ajuste o modelo com `statsmodels`.  
2. Interprete cada coeficiente.  
3. Verifique quais variáveis são significativas.  
4. Compare com os modelos simples.  

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 5 — Regressão no lugar da correlação

Teste a relação entre `beauty` e `eval` usando regressão.

1. Ajuste o modelo simples.  
2. Compare o valor‑p com o teste de Pearson.  
3. Compare R² com r².  
4. Plote o scatter + reta.  

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 6 — Previsões com regressão

Use o modelo `eval ~ beauty`.

1. Gere previsões para beauty = 0, 1, 2, 3.  
2. Plote os valores previstos.  
3. Interprete o significado prático.  

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 7 — Análise de resíduos

Use o modelo `eval ~ beauty`.

1. Plote resíduos vs beauty.  
2. Plote histograma dos resíduos.  
3. Avalie:
   - linearidade  
   - homocedasticidade  
   - normalidade  

4. Escreva uma conclusão.  

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 8 — Comparando modelos

Ajuste os modelos:

- M1: `eval ~ beauty`  
- M2: `eval ~ beauty + female_dummy`  
- M3: `eval ~ beauty + female_dummy + age`  

1. Compare R² e R² ajustado.  
2. Compare significância dos coeficientes.  
3. Escolha o melhor modelo e justifique.  

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 9 — Interpretação conceitual

Responda em texto:

1. O que significa β₁ em regressão simples?  
2. O que significa β₁ em regressão com dummy?  
3. Por que regressão substitui o teste t?  
4. Por que regressão substitui ANOVA?  
5. Por que regressão substitui correlação?  
6. O que significa R²?  
7. O que significa o termo de erro ε?  

In [ ]:
# TODO: escreva suas respostas aqui (em markdown)

---
# 📝 Exercício 10 — Projeto final do módulo 6

Escolha **qualquer variável Y contínua** e **duas variáveis X** do dataset.

1. Determine se X são contínuas ou categóricas.  
2. Crie dummies se necessário.  
3. Ajuste o modelo múltiplo:

$$
Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \varepsilon
$$

4. Interprete cada coeficiente.  
5. Avalie significância.  
6. Avalie resíduos.  
7. Escreva uma conclusão estatística.  

In [ ]:
# TODO: seu código aqui

<a id="apendice"></a>
# 8. Apêndice Matemático Avançado — Regressão Linear

Este apêndice aprofunda a teoria por trás da regressão linear:

- mínimos quadrados ordinários (OLS)  
- solução matricial  
- distribuição dos coeficientes  
- relação com correlação  
- relação com teste t  
- relação com ANOVA  
- R² e R² ajustado  
- variância dos coeficientes  
- multicolinearidade  

Tudo aqui é opcional, mas extremamente útil para dominar regressão.

---
# 8.1 O modelo linear

O modelo de regressão linear simples é:

$$
Y = \beta_0 + \beta_1 X + \varepsilon
$$

O modelo múltiplo é:

$$
Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_k X_k + \varepsilon
$$

Em forma matricial:

$$
\mathbf{Y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon}
$$

---
# 8.2 Mínimos Quadrados Ordinários (OLS)

O método OLS escolhe $\beta$ que minimiza:

$$
\sum_{i=1}^n (Y_i - \hat{Y}_i)^2
$$

A solução matricial é:

$$
\hat{\beta} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{Y}
$$

<details>
<summary>🟠 Demonstração — derivando a solução OLS</summary>

1. Escrevemos o erro quadrático:

$$
S(\beta) = (Y - X\beta)^\top (Y - X\beta)
$$

2. Derivamos em relação a β:

$$
\frac{\partial S}{\partial \beta} = -2X^\top(Y - X\beta)
$$

3. Igualamos a zero:

$$
X^\top Y = X^\top X \beta
$$

4. Resolvemos:

$$
\hat{\beta} = (X^\top X)^{-1} X^\top Y
$$

</details>

---
# 8.3 Distribuição dos coeficientes

Sob hipóteses clássicas:

$$
\hat{\beta} \sim N(\beta, \sigma^2 (X^\top X)^{-1})
$$

Isso permite:

- construir intervalos de confiança  
- realizar testes t  
- realizar testes F  

---
# 8.4 Teste t na regressão

A regressão testa automaticamente:

$$
H_0: \beta_j = 0
$$

A estatística t é:

$$
t = \frac{\hat{\beta}_j}{\text{SE}(\hat{\beta}_j)}
$$

Isso é exatamente o mesmo teste t do módulo 5.

---
# 8.5 Teste F na regressão

A regressão também testa:

$$
H_0: \beta_1 = \beta_2 = \dots = \beta_k = 0
$$

A estatística F é:

$$
F = \frac{MS_{\text{modelo}}}{MS_{\text{erro}}}
$$

Isso é exatamente o mesmo F da ANOVA.

---
# 8.6 Relação entre regressão e correlação

Para regressão simples:

$$
\beta_1 = r \cdot \frac{s_y}{s_x}
$$

E:

$$
R^2 = r^2
$$

<details>
<summary>🟠 Demonstração — por que R² = r²?</summary>

Em regressão simples:

$$
R^2 = \frac{\text{variância explicada}}{\text{variância total}}
$$

E:

$$
r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}
{\sqrt{\sum (x_i - \bar{x})^2} \sqrt{\sum (y_i - \bar{y})^2}}
$$

Substituindo β₁ na fórmula de R², obtemos:

$$
R^2 = r^2
$$

</details>

---
# 8.7 R² e R² ajustado

- **R²** mede proporção da variância explicada.  
- **R² ajustado** penaliza modelos com muitas variáveis.

Fórmula:

$$
R^2_{\text{adj}} = 1 - (1 - R^2)\frac{n - 1}{n - k - 1}
$$

---
# 8.8 Variância dos coeficientes

A variância de $\hat{\beta}$ é:

$$
\text{Var}(\hat{\beta}) = \sigma^2 (X^\top X)^{-1}
$$

Isso explica:

- por que multicolinearidade aumenta SE  
- por que coeficientes deixam de ser significativos  

---
# 8.9 Multicolinearidade

Ocorre quando variáveis explicativas são altamente correlacionadas.

Consequências:

- coeficientes instáveis  
- erros‑padrão inflados  
- testes t deixam de ser significativos  
- R² continua alto  

Indicadores:

- VIF (Variance Inflation Factor)  
- correlação alta entre colunas de X  

---
# 8.10 Interpretação geométrica da regressão

A regressão projeta Y no subespaço gerado pelas colunas de X.

- $\hat{Y}$ é a projeção ortogonal  
- os resíduos são ortogonais ao espaço de X  

Formalmente:

$$
X^\top (Y - \hat{Y}) = 0
$$

<details>
<summary>🟠 Demonstração — resíduos são ortogonais a X</summary>

Da equação normal:

$$
X^\top(Y - X\hat{\beta}) = 0
$$

Mas:

$$
X\hat{\beta} = \hat{Y}
$$

Logo:

$$
X^\top(Y - \hat{Y}) = 0
$$

Isso significa que os resíduos são perpendiculares ao espaço das variáveis explicativas.

</details>

---
# 8.11 Decomposição da variância

A regressão decompõe a variância total:

$$
SST = SSR + SSE
$$

Onde:

- SST = soma total dos quadrados  
- SSR = soma dos quadrados explicados  
- SSE = soma dos quadrados do erro  

---
# 8.12 Resumo visual — Normal, t e F

In [ ]:
import numpy as np
from scipy.stats import norm, t, f

x = np.linspace(-4, 4, 400)

plt.figure(figsize=(8, 4))
plt.plot(x, norm.pdf(x), label="Normal", linewidth=2)
plt.plot(x, t.pdf(x, df=5), label="t (df=5)")
plt.plot(x, t.pdf(x, df=30), label="t (df=30)")
plt.title("Comparação: Normal vs t-distribution")
plt.legend()
plt.grid(alpha=0.3)
plt.show()